In [1]:
print("""
@File         : selecting_the_highest_rated_movies_by_year.ipynb
@Author(s)    : Stephen CUI
@LastEditor(s): Stephen CUI
@CreatedTime  : 2025-01-05 14:09:36
@Email        : cuixuanstephen@gmail.com
@Description  : 按年份选择评分最高的电影
""")


@File         : selecting_the_highest_rated_movies_by_year.ipynb
@Author(s)    : Stephen CUI
@LastEditor(s): Stephen CUI
@CreatedTime  : 2025-01-05 14:09:36
@Email        : cuixuanstephen@gmail.com
@Description  : 按年份选择评分最高的电影



In [2]:
import pandas as pd

In [3]:
df = pd.read_csv(
    '../DATA/movie.csv',
    usecols=['movie_title', 'title_year', 'imdb_score'],
    dtype_backend='numpy_nullable'
)

df

,movie_title,title_year,imdb_score
0,Avatar,2009.0,7.9
1,Pirates of the Caribbean: At World's End,2007.0,7.1
2,Spectre,2015.0,6.8
3,The Dark Knight Rises,2012.0,8.5
4,Star Wars: Episode VII - The Force Awakens,<NA>,7.1
...,...,...,...
4911,Signed Sealed Delivered,2013.0,7.7
4912,The Following,<NA>,7.5
4913,A Plague So Pleasant,2013.0,6.3
4914,Shanghai Calling,2012.0,6.3


In [4]:
df['title_year'] =  df['title_year'].astype(pd.Int64Dtype())
df.head()

,movie_title,title_year,imdb_score
0,Avatar,2009,7.9
1,Pirates of the Caribbean: At World's End,2007,7.1
2,Spectre,2015,6.8
3,The Dark Knight Rises,2012,8.5
4,Star Wars: Episode VII - The Force Awakens,<NA>,7.1


In [6]:
df = pd.read_csv(
    '../DATA/movie.csv',
    usecols=['movie_title', 'title_year', 'imdb_score'],
    dtype={'title_year': pd.Int64Dtype()},
    dtype_backend='numpy_nullable'
)

df.head(3)

,movie_title,title_year,imdb_score
0,Avatar,2009,7.9
1,Pirates of the Caribbean: At World's End,2007,7.1
2,Spectre,2015,6.8


In [10]:
df.sort_values(['title_year', 'imdb_score']).groupby(
    'title_year'
)[['movie_title']].agg(top_rated_movie=pd.NamedAgg('movie_title', 'last'))

,top_rated_movie
title_year,
1916,Intolerance: Love's Struggle Throughout the Ages
1920,Over the Hill to the Poorhouse
1925,The Big Parade
1927,Metropolis
1929,Pandora's Box
...,...
2012,Django Unchained
2013,"Batman: The Dark Knight Returns, Part 2"
2014,Butterfly Girl


In [15]:
df.set_index('movie_title').groupby('title_year').agg(
    top_rated_movie=pd.NamedAgg('imdb_score', 'idxmax')
)

,top_rated_movie
title_year,
1916,Intolerance: Love's Struggle Throughout the Ages
1920,Over the Hill to the Poorhouse
1925,The Big Parade
1927,Metropolis
1929,Pandora's Box
...,...
2012,The Dark Knight Rises
2013,"Batman: The Dark Knight Returns, Part 2"
2014,Queen of the Mountains


> 这两种方法在结果上略微有些不同，关键就是在同一年的得分最高的电影是有多部相同得分的情况。第一种按照排序，排在最后的是年度电影。第二种是谁先出现在 index 中谁是年度电影。

如果出现平局，每种方法都有自己的选择值的方式。这两种方法本身没有对错之分，但如果你想对此进行更精细的控制，你就必须使用 `Group by apply`。

假设我们想要聚合这些值，这样当没有平局时，我们会返回一个字符串，但如果有平局，我们会得到一个字符串序列。为此，应该定义一个接受 pd.DataFrame 的函数。此 pd.DataFrame 将包含与每个唯一分组列（在我们的例子中为title_year）相关的值。

In [21]:
def top_rated(df: pd.DataFrame):
    top_rating = df['imdb_score'].max()
    top_rated = df[df['imdb_score'] == top_rating]['movie_title'].unique()
    
    if len(top_rated) == 1:
        return top_rated[0]
    else:
        return top_rated
    
    
df.groupby('title_year').apply(
    top_rated, include_groups=False
).to_frame().rename(columns={0: 'top_rated_movie(s)'})

,top_rated_movie(s)
title_year,
1916,Intolerance: Love's Struggle Throughout the Ages
1920,Over the Hill to the Poorhouse
1925,The Big Parade
1927,Metropolis
1929,Pandora's Box
...,...
2012,"[The Dark Knight Rises, Django Unchained]"
2013,"Batman: The Dark Knight Returns, Part 2"
2014,"[Queen of the Mountains, Butterfly Girl]"
